# Funnel Daily Report &rarr; Google Sheets

Builds a daily summary per project and writes it to that project's Google Sheet:

| Date | Sales | Core order | Upsell order | Total Order | Refund count | Refund amount | Refund rate | Spend | ROAS | CPA |

**How to use**
1. Edit the **Control cell** below (orders file, optional refunds file, optional ad-spend CSVs + one block per project).
2. Run every cell top-to-bottom (*Runtime &rarr; Run all*).
3. On first sign-in a popup asks for your Google account &mdash; click **Allow**.

**Append logic (per sheet tab)** &mdash; empty tab &rarr; writes the header + all dates;
tab with data &rarr; appends only dates *newer* than the latest one already there.

**Refunds** &mdash; matched by the **same funnels** (a `Funnel` column, or the `(funnel)`
inside the `Product` text), bucketed by **Refunded Date**. `Refund count` = distinct Order
IDs that day; `Refund amount` = total refunded (positive); `Refund rate` = `Refund amount / Sales`.

**Ad spend** &mdash; one consolidated workbook (`AD_SPEND_FILE`) holds every platform/product,
each row tagged in a `Source.Name` column. Each project lists `ads_match` token(s); a row is
attached when its `Source.Name` contains a token (e.g. `Source.Name` of `...Akka_LCP_Facebook...`
&rarr; the project with `ads_match=["Akka LCP Facebook"]`; matching ignores spaces/underscores).
`Spend` is summed by day. `ROAS` and `CPA` come straight from the sheet by default
(`ADS_METRICS_SOURCE="file"`), or set `"report"` to compute `ROAS = Sales/Spend` and
`CPA = Spend / orders` from this report instead.

> To add a project: copy a `{...}` block in `PROJECTS`, set `name`, `sheet_url`, `funnels`,
> and (optionally) `ads_match`. Funnels also match their `-dup` / `-v1` / `-text` variants.


## 1. Control cell  &mdash;  *edit this*

In [ ]:
# ===================== CONTROL CELL =====================
# Edit the values here, then run all the cells below.

# --- Source export -------------------------------------------------
SOURCE_FILE  = "Orders-only.xlsx"   # orders file name (upload it in the next cell)
SOURCE_SHEET = None                  # None = first sheet, or e.g. "Project Report"

# --- Refund export (optional) -------------------------------------
REFUND_FILE  = "refunds only.xlsx"
REFUND_SHEET = "Daily Refunds"
EXCLUDE_TEST_REFUNDS = True          # refund export has no "Order Type" column, so
REFUND_TEST_PATTERN  = r"tester|skstests"   # test orders are matched by name / email

# --- Ad spend (optional) ------------------------------------------
# ONE consolidated workbook of daily ad metrics: every platform/product CSV
# stacked into one sheet, each row tagged with its original file in the
# "Source.Name" column. A project is attached to the rows whose Source.Name
# contains its "ads_match" token(s). Set AD_SPEND_FILE = None to skip ad spend.
AD_SPEND_FILE  = "TNT Spend - only.xlsx"
AD_SPEND_SHEET = "TNT Marketplace Spend"            # data tab; None = first sheet
ADS_METRICS_SOURCE = "file"    # "file"  -> use the sheet's own Spend / ROAS / CPA columns
                               # "report"-> Spend from sheet; ROAS = Sales/Spend;
                               #            CPA = Spend / orders (see ADS_CPA_BASIS)
ADS_CPA_BASIS = "Total Order"  # only for ADS_METRICS_SOURCE="report":
                               # "Total Order", "Core order", or "Upsell order"

# --- Which orders to include --------------------------------------
EXCLUDE_TEST_ORDERS      = True
INCLUDED_CHARGE_STATUSES = None      # None = all; or {"Charged"} for completed sales only

# Funnel-name modifiers matched in ANY order/combination after a base funnel.
MODIFIERS = ["dup", "text", "v[0-9]+"]

# --- Projects ------------------------------------------------------
# funnels = base funnel names (variants auto-matched). ads_match = filename
# token(s) for that project's ad CSV(s); omit or set None for no ad spend.
PROJECTS = [
    {
        "name": "SF",
        "sheet_url": "https://docs.google.com/spreadsheets/d/1dFH03PAWAzZo7PsVd9w_wAvTuUtzX0LMXRFUJyWuRj8/edit",
        "worksheet": None,
        "funnels": ["sf-vslfb", "sf-pdpfb", "sf-pdpfb_sub", "sf-tslfb"],
        "ads_match": ["Sofon Facebook"],     # -> AdsKpi_Sofon_Facebook_...
    },
    {
        "name": "VBP-FB",
        "sheet_url": "https://docs.google.com/spreadsheets/d/1CZvLbrMf0L8LjNVh17K9Be6g6dyA-7opPFfPHASWiqM/edit",
        "worksheet": None,
        "funnels": ["vbp-vslfb", "vbp-tslfb", "vbp-vslfb-text"],
        "ads_match": ["Vital BP Facebook"],  # -> AdsKpi_Vital_BP_Facebook_...
    },
    {
        "name": "VBP-Google",
        "sheet_url": "https://docs.google.com/spreadsheets/d/1_zx6BPLndqCYLMnNLr8NTWrjbj6yLVoMlTEZdV6In8Q/edit",  # <-- set VBP-Google sheet
        "worksheet": None,
        "funnels": ["vbp-gs", "vbp-vslyt", "vbp-tslyt"],
        "ads_match": ["Vital BP Google"],    # -> AdsKpi_Vital_BP_Google_...
    },
    {
        "name": "AKKA-FB",
        "sheet_url": "https://docs.google.com/spreadsheets/d/1F05RiGAU69fuRIeXbypXUqmrY_iSvvBI7Tgqod8CmrU/edit",  # <-- set AKKA-FB sheet
        "worksheet": None,
        "funnels": ["akk-vslfb", "akk-tslfb"],   # Facebook funnels (add more if needed)
        "ads_match": ["Akka LCP Facebook"],      # matches AdsKpi_Akka_LCP_Facebook_...csv
    },
    {
        "name": "AKKA-Google",
        "sheet_url": "https://docs.google.com/spreadsheets/d/1sUMq8atT7fC-c3GOgUZ2Bl1V7RbGstk-0wtVmoHZsnc/edit",  # <-- set AKKA-Google sheet
        "worksheet": None,
        "funnels": ["akk-vslyt", "akk-gs", "akk-tslyt", "akk-vslyt-control", "sb-akk-vslyt"],
        "ads_match": ["Akka LCP Google"],    # -> AdsKpi_Akka_LCP_Google_...
    },
    {
        "name": "SYN-FB",
        "sheet_url": "https://docs.google.com/spreadsheets/d/1C0asFWldQ0zb_PNrglk0rbVtv1sVRrdun7X_2tEAmiY/edit",  # <-- set Synocell FB sheet
        "worksheet": None,
        "funnels": ["syn-vslfb", "syn-tslfb", "syn-vslfb-cro", "syn-pdpfb", "syn-pdpfb_sub"],
        "ads_match": ["Synocell Facebook"],  # -> AdsKpi_Synocell_Facebook_...
    },
    {
        "name": "SYN-Google",
        "sheet_url": "https://docs.google.com/spreadsheets/d/1ZQKbBtPPguppnhiqWH-IE4pxCuKDT3ecfjM5vKbDEsY/edit?gid=0#gid=0",  # <-- set Synocell Google sheet
        "worksheet": None,
        "funnels": ["syn-gs", "syn-vslyt", "syn-tslgg", "syn-ecom-gs", "syn-pdpyt", "syn-pdpyt_sub"],
        "ads_match": ["Synocell Google"],    # -> AdsKpi_Synocell_Google_...
    },
    {
        "name": "Skyhouse-FB",
        "sheet_url": "https://docs.google.com/spreadsheets/d/1u1OqQiFOmIjcxFOXzjw5TaqQ7tQJl5HrprWX1us-Ijo/edit?gid=0#gid=0",
        "worksheet": None,
        "funnels": ["lrp-akk-vslfb", "lrp-akk-vslfb-text"],
        "ads_match": None,   # <-- set ad-spend filename token if this project has ad spend
    },
    {
        "name": "Sarah-AKKA",
        "sheet_url": "https://docs.google.com/spreadsheets/d/1cNVbQYawVlghe319-VJ-LcLW1Nd803-Dw87Whq3AWgk/edit?gid=0#gid=0",
        "worksheet": None,
        "funnels": ["ryl-akk-pdpfb", "ryl-akk-vslfb", "ryl-akk-pdpfb_sub"],
        "ads_match": None,   # <-- set ad-spend filename token if this project has ad spend
    },
    {
        "name": "Sarah-Emma",
        "sheet_url": "https://docs.google.com/spreadsheets/d/1Jqojz_IZpdpV1WSjwRiw8AejwfaX62VeDXhatO0RgD0/edit?gid=0#gid=0",
        "worksheet": None,
        "funnels": ["grm-er-pdpfb", "grm-er-vslfb", "grm-er-pdpfb_sub"],
        "ads_match": None,   # <-- set ad-spend filename token if this project has ad spend
    },
    {
        "name": "Seneca",
        "sheet_url": "https://docs.google.com/spreadsheets/d/11-66AUzexBuNA_MSVyA4KMe5vHDq2eLNb5vZl9haZY8/edit?usp=sharing",
        "worksheet": None,
        "funnels": ["snc-vslfb"],   # base funnel; -dup/-v1/-text variants auto-match
        "ads_match": None,   # <-- set ad-spend filename token if this project has ad spend
    },

    # ---- template ----
    # {
    #     "name": "NEW-PROJECT",
    #     "sheet_url": "https://docs.google.com/spreadsheets/d/XXXX/edit",
    #     "worksheet": None,
    #     "funnels": ["xx-vslfb", "xx-tslfb"],
    #     "ads_match": ["XX LCP Facebook"],
    # },
]
# =======================================================
print(f"{len(PROJECTS)} project(s) configured: " + ", ".join(p["name"] for p in PROJECTS))

## 2. Setup  &mdash;  *run, don't edit*

In [ ]:
# Install dependencies (quiet). Safe to re-run.
!pip install gspread openpyxl -q
print("dependencies ready")


dependencies ready


In [ ]:
# ====================== ENGINE  (no need to edit) ======================
import os, re
from datetime import datetime, date, timedelta
from openpyxl import load_workbook

OUT_HEADERS  = ["Date", "Sales", "Core order", "Upsell order", "Total Order",
                "Refund count", "Refund amount", "Refund rate",
                "Spend", "ROAS", "CPA"]
DATE_STR_FMT = "%Y-%m-%d"

# Orders columns
COL_FUNNEL="Funnel"; COL_CUP="C/Up"; COL_ORDER_TYPE="Order Type"
COL_CHARGE="Charge Status"; COL_ORDER_DATE="Order Date"; COL_TOTAL_SALES="Total Sales"

# Refund columns (first matching spelling wins)
COLS_REFUND_FUNNEL  = ["Funnel"]
COLS_REFUND_PRODUCT = ["Product", "Product Name", "Product Title"]
COLS_REFUND_ORDERID = ["Order ID", "Order Id", "OrderID", "Order #",
                       "Order Number", "Order No", "Order"]
COLS_REFUND_DATE    = ["Refunded Date", "Refund Date", "Date Refunded",
                       "Refunded At", "Refund On", "Refund Time"]
COLS_REFUND_AMOUNT  = ["Refund Amount", "Refunded Amount", "Amount Refunded",
                       "Refund Total", "Total Refund", "Refunded"]
COLS_REFUND_NAME    = ["Customer Full Name", "Customer Name", "Full Name", "Name"]
COLS_REFUND_EMAIL   = ["Customer Email", "Email", "Customer E-mail"]
FUNNEL_IN_PRODUCT_RE = re.compile(r"\(([a-z0-9]+(?:[-_][a-z0-9]+)+)\)", re.IGNORECASE)

# Ad-spend CSV columns
COLS_ADS_DATE  = ["Day", "Date", "Reporting starts", "Reporting Day"]
COLS_ADS_SPEND = ["Spend", "Amount Spent", "Amount spent (USD)", "Cost"]
COLS_ADS_ROAS  = ["ROAS", "Purchase ROAS", "Website Purchase ROAS"]
COLS_ADS_CPA   = ["CPA", "Cost per Purchase", "Cost per Result"]
COLS_ADS_PURCH = ["Purchases", "Results", "Conversions"]
COLS_ADS_SOURCE = ["Source.Name", "Source Name", "Source", "File", "File Name"]


def _norm(s):
    return str(s).strip().lower() if s is not None else ""

def _loose(s):
    # separator-insensitive: spaces, underscores, hyphens, dots all equivalent
    return re.sub(r"[^a-z0-9]+", " ", str(s).lower()).strip()

def _to_date(v):
    if v in (None, ""): return None
    if isinstance(v, datetime): return v.date()
    if isinstance(v, date): return v
    s = str(v).strip()
    for fmt in ("%Y-%m-%d","%m/%d/%Y","%m-%d-%Y","%d/%m/%Y","%Y/%m/%d","%m/%d/%y"):
        try: return datetime.strptime(s, fmt).date()
        except ValueError: continue
    return None

def _to_number(v):
    if v in (None, ""): return 0.0
    if isinstance(v, (int, float)): return float(v)
    try: return float(str(v).replace(",", "").replace("$", "").strip())
    except ValueError: return 0.0

def _to_refund_amount(v):
    if v in (None, ""): return 0.0
    return abs(_to_number(str(v).replace("(", "-").replace(")", "")))

def _funnel_from_product(text):
    if text is None: return None
    m = FUNNEL_IN_PRODUCT_RE.findall(str(text))
    return m[-1].strip() if m else None

def find_header_index(header_row, name, required=True):
    target = _norm(name)
    for i, cell in enumerate(header_row):
        if _norm(cell) == target: return i
    if required:
        raise KeyError(f"Column '{name}' not found. Headers: "
                       f"{[c for c in header_row if c is not None]}")
    return None

def find_header_index_any(header_row, names, required=True, label=None):
    for nm in names:
        idx = find_header_index(header_row, nm, required=False)
        if idx is not None: return idx
    if required:
        raise KeyError(f"None of the {label or ''} columns {names} were found. "
                       f"Headers present: {[c for c in header_row if c is not None]}")
    return None

def build_pattern(funnels, modifiers=None):
    bases = sorted({f.strip() for f in funnels}, key=len, reverse=True)
    alt = "|".join(re.escape(b) for b in bases)
    suffix = rf"(?:-(?:{'|'.join(modifiers)}))*" if modifiers else ""
    return re.compile(rf"^(?:{alt}){suffix}$", re.IGNORECASE)


def aggregate_source(source_path, source_sheet, projects):
    """Return {project_name: {date: [sales, core, upsell, total]}}."""
    wb = load_workbook(source_path, data_only=True, read_only=True)
    ws = wb[source_sheet] if source_sheet else wb[wb.sheetnames[0]]
    rows = ws.iter_rows(values_only=True)
    header = list(next(rows))
    i_funnel = find_header_index(header, COL_FUNNEL)
    i_cup    = find_header_index(header, COL_CUP)
    i_date   = find_header_index(header, COL_ORDER_DATE)
    i_sales  = find_header_index(header, COL_TOTAL_SALES)
    i_otype  = find_header_index(header, COL_ORDER_TYPE, required=False)
    i_charge = find_header_index(header, COL_CHARGE, required=False)
    pats = {p["name"]: build_pattern(p["funnels"], MODIFIERS) for p in projects}
    agg  = {p["name"]: {} for p in projects}
    allowed = ({_norm(s) for s in INCLUDED_CHARGE_STATUSES}
               if INCLUDED_CHARGE_STATUSES is not None else None)
    for row in rows:
        if not row: continue
        funnel = row[i_funnel] if i_funnel < len(row) else None
        if funnel is None: continue
        fstr = str(funnel).strip()
        matched = [name for name, pat in pats.items() if pat.match(fstr)]
        if not matched: continue
        if EXCLUDE_TEST_ORDERS and i_otype is not None and _norm(row[i_otype]) == "test order":
            continue
        if allowed is not None and i_charge is not None and _norm(row[i_charge]) not in allowed:
            continue
        d = _to_date(row[i_date] if i_date < len(row) else None)
        if d is None: continue
        sales = _to_number(row[i_sales] if i_sales < len(row) else 0)
        cup = _norm(row[i_cup] if i_cup < len(row) else None)
        for name in matched:
            b = agg[name].setdefault(d, [0.0, 0, 0, 0])
            b[0] += sales
            if cup == "core": b[1] += 1
            elif cup == "upsell": b[2] += 1
            b[3] += 1
    wb.close()
    return agg


def aggregate_refunds(refund_path, refund_sheet, projects):
    """Return {project_name: {date: [refund_order_count, refund_amount]}}.
    Bucketed by REFUNDED DATE; count = distinct Order IDs."""
    if not refund_path:
        return {p["name"]: {} for p in projects}
    wb = load_workbook(refund_path, data_only=True, read_only=True)
    ws = wb[refund_sheet] if refund_sheet else wb[wb.sheetnames[0]]
    rows = ws.iter_rows(values_only=True)
    header = list(next(rows))
    i_funnel  = find_header_index_any(header, COLS_REFUND_FUNNEL,  required=False)
    i_product = find_header_index_any(header, COLS_REFUND_PRODUCT, required=False)
    if i_funnel is None and i_product is None:
        raise KeyError("Refund file needs a 'Funnel' column or a 'Product' column "
                       f"with the funnel in parentheses. Headers: "
                       f"{[c for c in header if c is not None]}")
    i_oid   = find_header_index_any(header, COLS_REFUND_ORDERID, label="order id")
    i_date  = find_header_index_any(header, COLS_REFUND_DATE,    label="refund date")
    i_amt   = find_header_index_any(header, COLS_REFUND_AMOUNT,  label="refund amount")
    i_name  = find_header_index_any(header, COLS_REFUND_NAME,  required=False)
    i_email = find_header_index_any(header, COLS_REFUND_EMAIL, required=False)
    test_re = (re.compile(REFUND_TEST_PATTERN, re.IGNORECASE)
               if EXCLUDE_TEST_REFUNDS and REFUND_TEST_PATTERN else None)
    pats = {p["name"]: build_pattern(p["funnels"], MODIFIERS) for p in projects}
    ids  = {p["name"]: {} for p in projects}
    amt  = {p["name"]: {} for p in projects}
    for rownum, row in enumerate(rows, start=2):
        if not row: continue
        funnel = row[i_funnel] if (i_funnel is not None and i_funnel < len(row)) else None
        if not funnel and i_product is not None and i_product < len(row):
            funnel = _funnel_from_product(row[i_product])
        if funnel is None: continue
        matched = [name for name, pat in pats.items() if pat.match(str(funnel).strip())]
        if not matched: continue
        if test_re is not None:
            blob = " ".join(str(row[i]) for i in (i_name, i_email)
                            if i is not None and i < len(row) and row[i] is not None)
            if blob and test_re.search(blob): continue
        d = _to_date(row[i_date] if i_date < len(row) else None)
        if d is None: continue
        oid_raw = row[i_oid] if i_oid < len(row) else None
        oid = str(oid_raw).strip() if oid_raw not in (None, "") else f"__row{rownum}"
        a = _to_refund_amount(row[i_amt] if i_amt < len(row) else 0)
        for name in matched:
            ids[name].setdefault(d, set()).add(oid)
            amt[name][d] = amt[name].get(d, 0.0) + a
    wb.close()
    out = {}
    for name in ids:
        out[name] = {d: [len(ids[name][d]), round(amt[name].get(d, 0.0), 2)]
                     for d in ids[name]}
    return out


def aggregate_ads(ad_spend_file, ad_spend_sheet, projects):
    """Return {project_name: {date: {'spend','roas','cpa'}}}.
    One consolidated workbook; each row is attached to a project when one of that
    project's ads_match token(s) appears in the row's Source.Name (separator-
    insensitive, so "Vital BP Facebook" matches "Vital_BP_Facebook")."""
    if not ad_spend_file:
        return {p["name"]: {} for p in projects}

    wb = load_workbook(ad_spend_file, data_only=True, read_only=True)
    ws = wb[ad_spend_sheet] if ad_spend_sheet else wb[wb.sheetnames[0]]
    rows = ws.iter_rows(values_only=True)
    header = list(next(rows))

    i_src   = find_header_index_any(header, COLS_ADS_SOURCE, label="ad source")
    i_date  = find_header_index_any(header, COLS_ADS_DATE,   label="ads date")
    i_spend = find_header_index_any(header, COLS_ADS_SPEND,  label="ads spend")
    i_roas  = find_header_index_any(header, COLS_ADS_ROAS,   required=False)
    i_cpa   = find_header_index_any(header, COLS_ADS_CPA,    required=False)
    i_purch = find_header_index_any(header, COLS_ADS_PURCH,  required=False)

    toks = {}
    for p in projects:
        t = p.get("ads_match") or []
        if isinstance(t, str): t = [t]
        toks[p["name"]] = [_loose(x) for x in t if x]

    acc = {p["name"]: {} for p in projects}   # name -> date -> [spend, purch, adrev]
    for row in rows:
        if not row:
            continue
        src = row[i_src] if i_src < len(row) else None
        if src is None:
            continue
        srcl = _loose(src)
        names = [nm for nm, tl in toks.items() if tl and any(t in srcl for t in tl)]
        if not names:
            continue
        d = _to_date(row[i_date] if i_date < len(row) else None)
        if d is None:
            continue
        spend = _to_number(row[i_spend] if i_spend < len(row) else 0)
        roas  = _to_number(row[i_roas]) if (i_roas is not None and i_roas < len(row)) else 0.0
        cpa   = _to_number(row[i_cpa])  if (i_cpa  is not None and i_cpa  < len(row)) else 0.0
        if i_purch is not None and i_purch < len(row):
            purch = _to_number(row[i_purch])
        elif cpa > 0:
            purch = spend / cpa                 # reconstruct when no Purchases column
        else:
            purch = 0.0
        adrev = roas * spend                    # reconstruct ad-attributed revenue
        for nm in names:
            cur = acc[nm].setdefault(d, [0.0, 0.0, 0.0])
            cur[0] += spend; cur[1] += purch; cur[2] += adrev
    wb.close()

    out = {}
    for nm, dd in acc.items():
        out[nm] = {}
        for d, (spend, purch, adrev) in dd.items():
            roas = (adrev / spend) if spend else 0.0
            cpa  = (spend / purch) if purch else 0.0
            out[nm][d] = {"spend": round(spend, 2),
                          "roas": round(roas, 2),
                          "cpa": round(cpa, 2)}
    return out


def build_rows(order_buckets, refund_buckets, ads_buckets, dates, start_row):
    out = []
    r = start_row
    cpa_col = {"total order": "E", "core order": "C",
               "upsell order": "D"}.get(_norm(ADS_CPA_BASIS), "E")
    report_mode = (_norm(ADS_METRICS_SOURCE) == "report")
    for d in dates:
        sales, core, upsell, _t = order_buckets.get(d, [0.0, 0, 0, 0])
        rcount, ramount = refund_buckets.get(d, [0, 0.0])
        ad = ads_buckets.get(d)
        if ad is None:
            spend_cell = roas_cell = cpa_cell = ""
        else:
            spend_cell = round(ad["spend"], 2)
            if report_mode:
                roas_cell = f'=IF(I{r}=0,"",B{r}/I{r})'
                cpa_cell  = f'=IF({cpa_col}{r}=0,"",I{r}/{cpa_col}{r})'
            else:
                roas_cell = f'=IF(I{r}=0,"",B{r}/I{r})' if ad["spend"] else ""
                cpa_cell  = round(ad["cpa"], 2) if ad["cpa"] else ""
        out.append([d.strftime(DATE_STR_FMT), round(sales, 2), core, upsell,
                    f"=C{r}+D{r}", rcount, round(ramount, 2),
                    f'=IF(B{r}=0,"",G{r}/B{r})',
                    spend_cell, roas_cell, cpa_cell])
        r += 1
    return out


def _canon(v):
    """Canonicalize a cell for change-detection: numbers compare numerically
    (11734.2 == "11734.20"), everything else as a stripped string."""
    if v is None or v == "":
        return ""
    if isinstance(v, (int, float)):
        return round(float(v), 4)
    s = str(v).strip()
    try:
        return round(float(s.replace(",", "").replace("$", "")), 4)
    except ValueError:
        return s


def _sheet_date(v):
    """Parse a date from a Google Sheets cell robustly. When the grid is read
    UNFORMATTED, real date values come back as serial numbers (days since
    1899-12-30); text dates come back as strings in any locale format."""
    if v is None or v == "" or isinstance(v, bool):
        return None
    if isinstance(v, (int, float)):
        try:
            return (datetime(1899, 12, 30) + timedelta(days=int(v))).date()
        except (ValueError, OverflowError):
            return None
    return _to_date(v)          # string path -> existing multi-format parser


def _expected_values(d, order_buckets, refund_buckets, ads_buckets):
    """The effective (formula-evaluated) values a row for date d should hold.
    Mirrors build_rows exactly, but returns the computed numbers instead of
    the "=..." formula strings, so we can compare against what the sheet is
    actually showing and catch when a derived column (e.g. ROAS after its
    definition changed) needs rewriting -- not just when raw inputs change.
    Keep this in lockstep with build_rows."""
    sales, core, upsell, _t = order_buckets.get(d, [0.0, 0, 0, 0])
    rcount, ramount = refund_buckets.get(d, [0, 0.0])
    b_sales = round(sales, 2)
    g_ref   = round(ramount, 2)
    total   = core + upsell
    rate    = (g_ref / b_sales) if b_sales else ""
    ad = ads_buckets.get(d)
    if ad is None:
        spend = roas = cpa = ""
    else:
        spend = round(ad["spend"], 2)
        roas  = (b_sales / spend) if spend else ""
        if _norm(ADS_METRICS_SOURCE) == "report":
            basis = {"total order": total, "core order": core,
                     "upsell order": upsell}.get(_norm(ADS_CPA_BASIS), total)
            cpa = (spend / basis) if basis else ""
        else:
            cpa = round(ad["cpa"], 2) if ad["cpa"] else ""
    return [d.strftime(DATE_STR_FMT), b_sales, core, upsell, total,
            rcount, g_ref, rate, spend, roas, cpa]


def _values_equal(expected, existing_row):
    """Compare the effective values a row should have against what the sheet
    currently stores. Numbers compare numerically (tolerant of rounding/format);
    the Date column (index 0) is skipped since rows are matched by date."""
    for i, exp in enumerate(expected):
        if i == 0:
            continue
        cur = existing_row[i] if i < len(existing_row) else ""
        if _canon(exp) != _canon(cur):
            return False
    return True


def write_gsheet(project, order_buckets, refund_buckets, ads_buckets, client):
    import gspread
    sh = client.open_by_url(project["sheet_url"])
    ws = sh.sheet1 if not project.get("worksheet") else sh.worksheet(project["worksheet"])
    # Read UNFORMATTED so dates come back as deterministic serial numbers
    # instead of a locale-dependent display string that we can't reliably parse.
    existing  = ws.get_all_values(
        value_render_option=gspread.utils.ValueRenderOption.unformatted)
    data_rows = [row for row in existing if any(str(c).strip() for c in row)]
    has_header = bool(data_rows) and _norm(data_rows[0][0]) == _norm(OUT_HEADERS[0])
    has_data   = len(data_rows) > (1 if has_header else 0)
    all_dates = sorted(set(order_buckets) | set(refund_buckets) | set(ads_buckets))

    if has_header and len(existing[0]) < len(OUT_HEADERS):
        ws.update(values=[OUT_HEADERS], range_name="A1", value_input_option="USER_ENTERED")

    # Empty sheet -> write header + every row.
    if not has_data:
        payload = [OUT_HEADERS] + build_rows(order_buckets, refund_buckets, ads_buckets,
                                             all_dates, start_row=2)
        ws.update(values=payload, range_name="A1", value_input_option="USER_ENTERED")
        return ("created", all_dates, None)

    # Map each date already in the sheet to its absolute (1-based) row number.
    date_to_row = {}
    unparsed, dupes = [], []
    for idx, row in enumerate(existing):
        if has_header and idx == 0:            # skip header
            continue
        if not row or str(row[0]).strip() == "":
            continue
        d = _sheet_date(row[0])
        if d is None:
            unparsed.append((idx + 1, row[0]))
        elif d in date_to_row:
            dupes.append((d, date_to_row[d], idx + 1))
        else:
            date_to_row[d] = idx + 1           # sheet rows are 1-based (first wins)
    if unparsed:
        print(f"[{project['name']}] WARNING: {len(unparsed)} sheet row(s) had an "
              f"unreadable Date and can't be matched (will be treated as new). "
              f"e.g. row {unparsed[0][0]} = {unparsed[0][1]!r}")
    if dupes:
        print(f"[{project['name']}] WARNING: {len(dupes)} duplicate date(s) already "
              f"in the sheet; overwriting the first occurrence only. e.g. {dupes[0][0]} "
              f"in rows {dupes[0][1]} and {dupes[0][2]}")
    latest = max(date_to_row) if date_to_row else None

    # Partition: existing dates whose data changed -> overwrite in place;
    #            dates not in the sheet          -> append at the bottom.
    changed_rows = []          # (sheet_row, date, built_row)
    new_dates    = []
    unchanged    = 0
    for d in all_dates:
        r = date_to_row.get(d)
        if r is None:
            new_dates.append(d)
            continue
        expected = _expected_values(d, order_buckets, refund_buckets, ads_buckets)
        if _values_equal(expected, existing[r - 1]):
            unchanged += 1
        else:
            built = build_rows(order_buckets, refund_buckets, ads_buckets, [d], start_row=r)[0]
            changed_rows.append((r, d, built))

    # Overwrite changed rows in one batched request. Each row is rebuilt for
    # its own row number so its formula cells reference the correct line.
    if changed_rows:
        ws.batch_update(
            [{"range": f"A{r}", "values": [built]} for r, _d, built in changed_rows],
            value_input_option="USER_ENTERED")

    # Append brand-new dates as a contiguous block under the last used row.
    if new_dates:
        start = len(existing) + 1
        ws.update(values=build_rows(order_buckets, refund_buckets, ads_buckets,
                                    new_dates, start_row=start),
                  range_name=f"A{start}", value_input_option="USER_ENTERED")

    info = {"updated":  sorted(d for _r, d, _b in changed_rows),
            "appended": sorted(new_dates),
            "unchanged": unchanged}
    if not changed_rows and not new_dates:
        return ("up_to_date", info, latest)
    return ("synced", info, latest)


def get_colab_client():
    from google.colab import auth
    auth.authenticate_user()
    import gspread
    from google.auth import default
    creds, _ = default()
    return gspread.authorize(creds)


def run(projects, source, client, source_sheet=None,
        refund=None, refund_sheet=None,
        ad_spend_file=None, ad_spend_sheet=None, only=None):
    if only:
        projects = [p for p in projects if p["name"].lower() == only.lower()]
        if not projects: raise ValueError(f"No project named '{only}'.")
    agg  = aggregate_source(source, source_sheet, projects)
    ragg = aggregate_refunds(refund, refund_sheet, projects)
    aagg = aggregate_ads(ad_spend_file, ad_spend_sheet, projects)
    for p in projects:
        ob = agg.get(p["name"], {}); rb = ragg.get(p["name"], {}); ab = aagg.get(p["name"], {})
        if not ob and not rb and not ab:
            print(f"[{p['name']}] no matching rows in orders, refunds or ads. Skipped.")
            continue
        status, info, latest = write_gsheet(p, ob, rb, ab, client)
        if status == "created":
            print(f"[{p['name']}] Built fresh {len(info)} row(s): {info[0]} -> {info[-1]}")
        elif status == "up_to_date":
            print(f"[{p['name']}] already up to date (latest {latest}). "
                  f"{info['unchanged']} row(s) checked, nothing changed.")
        else:  # synced
            upd, app = info["updated"], info["appended"]
            parts = []
            if app: parts.append(f"appended {len(app)} new ({app[0]} -> {app[-1]})")
            if upd: parts.append(f"overwrote {len(upd)} changed ({upd[0]} -> {upd[-1]})")
            tail = f"; {info['unchanged']} unchanged" if info["unchanged"] else ""
            print(f"[{p['name']}] " + "; ".join(parts) + tail +
                  (f" (latest {latest})" if latest else ""))

print("engine loaded")


engine loaded


## 3. Preview  &mdash;  *check the numbers before writing*

In [ ]:
# Dry run: aggregate orders + refunds + ad spend and show totals per project
# WITHOUT touching any sheet. Verifies funnel matches and which ad files attach.
import openpyxl as _ox

def _all_funnels_orders(path, sheet):
    wb=_ox.load_workbook(path,data_only=True,read_only=True)
    ws=wb[sheet] if sheet else wb[wb.sheetnames[0]]
    rows=ws.iter_rows(values_only=True); hdr=list(next(rows)); fi=find_header_index(hdr,COL_FUNNEL)
    found=set()
    for r in rows:
        if r and fi<len(r) and r[fi] is not None: found.add(str(r[fi]).strip())
    wb.close(); return found

def _all_funnels_refunds(path, sheet):
    wb=_ox.load_workbook(path,data_only=True,read_only=True)
    ws=wb[sheet] if sheet else wb[wb.sheetnames[0]]
    rows=ws.iter_rows(values_only=True); hdr=list(next(rows))
    fi=find_header_index_any(hdr,COLS_REFUND_FUNNEL,required=False)
    pi=find_header_index_any(hdr,COLS_REFUND_PRODUCT,required=False)
    found=set()
    for r in rows:
        if not r: continue
        f=r[fi] if (fi is not None and fi<len(r)) else None
        if not f and pi is not None and pi<len(r): f=_funnel_from_product(r[pi])
        if f: found.add(str(f).strip())
    wb.close(); return found

_all_orders  = _all_funnels_orders(SOURCE_FILE, SOURCE_SHEET)
_all_refunds = _all_funnels_refunds(REFUND_FILE, REFUND_SHEET) if REFUND_FILE else set()
_po = aggregate_source(SOURCE_FILE, SOURCE_SHEET, PROJECTS)
_pr = aggregate_refunds(REFUND_FILE, REFUND_SHEET, PROJECTS)
_pa = aggregate_ads(AD_SPEND_FILE, AD_SPEND_SHEET, PROJECTS)
def _all_sources(path, sheet):
    if not path: return []
    wb=_ox.load_workbook(path,data_only=True,read_only=True)
    ws=wb[sheet] if sheet else wb[wb.sheetnames[0]]
    rows=ws.iter_rows(values_only=True); hdr=list(next(rows))
    si=find_header_index_any(hdr,COLS_ADS_SOURCE)
    found=set()
    for r in rows:
        if r and si<len(r) and r[si] is not None: found.add(str(r[si]).strip())
    wb.close(); return sorted(found)
_sources = _all_sources(AD_SPEND_FILE, AD_SPEND_SHEET)

for p in PROJECTS:
    b=_po[p["name"]]; rb=_pr[p["name"]]; ab=_pa[p["name"]]
    pat=build_pattern(p["funnels"], MODIFIERS)
    print(f"[{p['name']}] order funnels:  {sorted(f for f in _all_orders if pat.match(f))}")
    if REFUND_FILE:
        print(f"[{p['name']}] refund funnels: {sorted(f for f in _all_refunds if pat.match(f))}")
    toks=p.get('ads_match') or []
    if isinstance(toks,str): toks=[toks]
    matched_src=[s for s in _sources
                 if any(_loose(t) in _loose(s) for t in toks)]
    if toks: print(f"[{p['name']}] ad sources:     {matched_src or 'NONE matched token(s) '+str(toks)}")
    tot_sales=round(sum(x[0] for x in b.values()),2) if b else 0.0
    print(f"   orders : dates={len(b)} sales={tot_sales} "
          f"core={sum(x[1] for x in b.values())} upsell={sum(x[2] for x in b.values())} "
          f"orders={sum(x[3] for x in b.values())}" if b else "   orders : no matching rows.")
    if rb:
        tr=round(sum(x[1] for x in rb.values()),2)
        rr=f"{(tr/tot_sales*100):.1f}%" if tot_sales else "n/a"
        print(f"   refunds: dates={len(rb)} count={sum(x[0] for x in rb.values())} amount={tr} rate={rr}")
    if ab:
        ts=round(sum(v['spend'] for v in ab.values()),2)
        broas=f"{(tot_sales/ts):.2f}" if ts else "n/a"
        print(f"   ads    : dates={len(ab)} spend={ts} | report-blended ROAS(Sales/Spend)={broas} "
              f"({min(ab)} -> {max(ab)})")
    print()


[SF] order funnels:  ['sf-pdpfb', 'sf-pdpfb_sub', 'sf-vslfb', 'sf-vslfb-dup', 'sf-vslfb-v1']
[SF] refund funnels: ['sf-pdpfb', 'sf-pdpfb_sub', 'sf-vslfb', 'sf-vslfb-dup', 'sf-vslfb-v1', 'sf-vslfb-v2', 'sf-vslfb-v3', 'sf-vslfb-v4']
[SF] ad sources:     ['AdsKpi_Sofon_Facebook_2026-06-01_to_2026-06-30_Day.csv']
   orders : dates=30 sales=248190.75 core=1343 upsell=231 orders=1574
   refunds: dates=30 count=206 amount=41669.93 rate=16.8%
   ads    : dates=30 spend=195828.09 | report-blended ROAS(Sales/Spend)=1.27 (2026-06-01 -> 2026-06-30)

[VBP-FB] order funnels:  ['vbp-tslfb', 'vbp-vslfb', 'vbp-vslfb-dup', 'vbp-vslfb-text', 'vbp-vslfb-v1', 'vbp-vslfb-v2']
[VBP-FB] refund funnels: ['vbp-tslfb', 'vbp-vslfb', 'vbp-vslfb-dup', 'vbp-vslfb-text', 'vbp-vslfb-v1', 'vbp-vslfb-v2']
[VBP-FB] ad sources:     ['AdsKpi_Vital_BP_Facebook_2026-06-01_to_2026-06-30_Day.csv']
   orders : dates=30 sales=913284.9 core=4956 upsell=1050 orders=6006
   refunds: dates=30 count=359 amount=63735.2 rate=7.0%
   ad

## 4. Sign in &amp; write to the sheets

In [ ]:
client = get_colab_client()          # popup the first time -> pick account, Allow
run(PROJECTS, SOURCE_FILE, client,
    refund=REFUND_FILE, refund_sheet=REFUND_SHEET,
    ad_spend_file=AD_SPEND_FILE, ad_spend_sheet=AD_SPEND_SHEET)
# run(..., only="AKKA-FB")   # to write just one project


[SF] overwrote 30 changed (2026-06-01 -> 2026-06-30) (latest 2026-06-30)
[VBP-FB] overwrote 30 changed (2026-06-01 -> 2026-06-30) (latest 2026-06-30)
[VBP-Google] overwrote 30 changed (2026-06-01 -> 2026-06-30) (latest 2026-06-30)
[AKKA-FB] overwrote 30 changed (2026-06-01 -> 2026-06-30) (latest 2026-06-30)
[AKKA-Google] overwrote 30 changed (2026-06-01 -> 2026-06-30) (latest 2026-06-30)
[SYN-FB] overwrote 30 changed (2026-06-01 -> 2026-06-30) (latest 2026-06-30)
[SYN-Google] overwrote 30 changed (2026-06-01 -> 2026-06-30) (latest 2026-06-30)


In [ ]:
# ==================== EXPORT: one CSV of everything, prefixed by platform ====================
# Re-aggregates the same files you just ran and stacks every project into a single
# CSV. Values match the sheets (ROAS = Sales/Spend, etc.). Run after the engine cell.
import csv

def export_csv(path="funnel_report_export.csv"):
    agg  = aggregate_source(SOURCE_FILE, SOURCE_SHEET, PROJECTS)
    ragg = aggregate_refunds(REFUND_FILE, REFUND_SHEET, PROJECTS)
    aagg = aggregate_ads(AD_SPEND_FILE, AD_SPEND_SHEET, PROJECTS)

    header = ["platform"] + OUT_HEADERS
    rows = []
    for p in PROJECTS:
        name = p["name"]
        ob, rb, ab = agg.get(name, {}), ragg.get(name, {}), aagg.get(name, {})
        for d in sorted(set(ob) | set(rb) | set(ab)):
            v = _expected_values(d, ob, rb, ab)          # numbers, not formulas
            v[9]  = round(v[9], 2)  if isinstance(v[9], (int, float))  else v[9]   # ROAS
            v[10] = round(v[10], 2) if isinstance(v[10], (int, float)) else v[10]  # CPA
            rows.append([name] + v)

    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(header)
        w.writerows(rows)

    print(f"wrote {len(rows)} row(s) across {len(PROJECTS)} platform(s) -> {path}\n")
    print(",".join(header))
    for r in rows:
        print(",".join("" if c == "" else str(c) for c in r))

    try:                                   # auto-download in Colab
        from google.colab import files
        files.download(path)
    except Exception:
        pass
    return path

export_csv()